## Woolworths Recipe Download for LLM Fine Tuning and DiscountMate Integration

**Aim**: Download all recipes for LLM integration to discountmate web platform. 

**Goal**: Train Q/A LLMs, General feedback LLMs, etc...

**Usage**: User asks "What kind of recipes can I make with cranberry sauce", "I feel like falafel tonight, what do you suggest?" --> LLM fine tuned on this recipe set AND the product data can suggest various recipe combos, what products to look at consolidate a shopping list for various recipes to try this week ETC... 

### Implementation:

**Note:** Automated docker run not set up for this, idea is to download 13,000 recipes, but there is provision for the code to download only new recipes, not sure an automated scripting run is required at this point but can be looked at. Hence for deployment just run in this jupyter book otherwise all code is self-contained and can be deployed as .py file....

### Process:

- Directory structure: All files saved in woolworths_recipes/
- JSON-LD only: Extracts only Recipe schema, saves as .json
- CSV tracker: recipe_tracker.csv tracks download status
- New recipe detection: Compares sitemap with existing tracker
- 4 iterations max: Loops up to 4 times for failed recipes
- Final iteration retries: Up to 5 retries with forced IP rotation on iteration 4
- ScraperAPI integration: All requests through ScraperAPI with rotation
- Auto-save progress: Every 5 recipes


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import json
import os
import time
import random
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import urllib3
from xml.etree import ElementTree

# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Global Settings
SCRAPERAPI_KEY = '96986696969979797977' # enter your ScraperAPI key here
OUTPUT_DIR = 'woolworths_recipes'
CSV_FILENAME = 'recipe_tracker.csv'
MAX_ITERATIONS = 4
MAX_RETRIES_ON_FINAL_ITERATION = 5

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/114.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.woolworths.com.au/",
}

def fetch_sitemap(url):
    """Fetch sitemap with proper headers"""
    try:
        session = requests.Session()
        session.headers.update(HEADERS)
        response = session.get(url, verify=False, timeout=30)
        
        if response.status_code == 200:
            return response.content
        else:
            print(f"WARNING: Status {response.status_code} for {url}")
            return None
            
    except Exception as e:
        print(f"ERROR: Error fetching {url}: {e}")
        return None

def parse_recipe_urls(content):
    """Extract recipe URLs from sitemap"""
    root = ElementTree.fromstring(content)
    ns = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
    
    recipes = []
    for url_elem in root.findall('.//ns:url', ns):
        loc = url_elem.find('ns:loc', ns)
        if loc is not None:
            url = loc.text
            # Extract recipe slug from URL
            recipe_slug = url.split('/')[-1]
            recipes.append({
                'recipe_slug': recipe_slug,
                'url': url,
                'downloaded': False,
                'download_status': '',
                'error_msg': '',
                'timestamp': None
            })
    
    return recipes

def load_or_create_tracker():
    """Load existing tracker CSV or create new one"""
    csv_path = os.path.join(OUTPUT_DIR, CSV_FILENAME)
    
    if os.path.exists(csv_path):
        print("INFO: Loading existing recipe tracker")
        df = pd.read_csv(csv_path)
        # Ensure boolean column is properly typed
        df['downloaded'] = df['downloaded'].fillna(False).astype(bool)
        return df
    else:
        print("INFO: No existing tracker found, will create new one")
        return None

def save_tracker(df):
    """Save tracker DataFrame to CSV"""
    csv_path = os.path.join(OUTPUT_DIR, CSV_FILENAME)
    try:
        df.to_csv(csv_path, index=False, encoding='utf-8')
        print(f"SUCCESS: Tracker saved to {csv_path}")
        return True
    except Exception as e:
        print(f"ERROR: Failed to save tracker: {e}")
        return False

def fetch_recipe_via_scraperapi(url, force_rotate=False):
    """Fetch recipe page using ScraperAPI"""
    try:
        params = {
            'api_key': SCRAPERAPI_KEY,
            'url': url,
        }
        
        if force_rotate:
            params['session_number'] = random.randint(1, 10000)
        
        response = requests.get('http://api.scraperapi.com', params=params, timeout=60)
        
        if response.status_code == 200:
            return response.text
        else:
            print(f"  WARNING: HTTP {response.status_code} from ScraperAPI")
            return None
            
    except Exception as e:
        print(f"  ERROR: ScraperAPI request failed: {type(e).__name__}")
        return None

def extract_json_ld(html):
    """Extract only JSON-LD Recipe schema from HTML"""
    try:
        soup = BeautifulSoup(html, 'html.parser')
        json_ld_scripts = soup.find_all('script', type='application/ld+json')
        
        for script in json_ld_scripts:
            try:
                data = json.loads(script.string)
                
                # Check if it's a Recipe schema
                if isinstance(data, dict) and data.get('@type') == 'Recipe':
                    return data, "SUCCESS"
                
                # Handle array of schemas
                if isinstance(data, list):
                    for item in data:
                        if isinstance(item, dict) and item.get('@type') == 'Recipe':
                            return item, "SUCCESS"
            except:
                continue
        
        return None, "NO_RECIPE_SCHEMA"
        
    except Exception as e:
        print(f"  ERROR: Parse error: {type(e).__name__}")
        return None, f"ERROR_{type(e).__name__}"

def save_recipe_json(recipe_data, recipe_slug):
    """Save recipe JSON-LD to file"""
    try:
        filename = f"{recipe_slug}.json"
        filepath = os.path.join(OUTPUT_DIR, filename)
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(recipe_data, f, indent=2, ensure_ascii=False)
        
        return True, filename
    except Exception as e:
        print(f"  ERROR: Failed to save JSON: {e}")
        return False, None

def scrape_single_recipe(url, recipe_slug, force_rotate=False):
    """Scrape a single recipe and extract JSON-LD"""
    try:
        html = fetch_recipe_via_scraperapi(url, force_rotate=force_rotate)
        
        if not html:
            return False, None, "SCRAPERAPI_FAILED"
        
        recipe_data, status = extract_json_ld(html)
        
        if not recipe_data:
            print(f"  FAIL: {status}")
            return False, None, status
        
        success, filename = save_recipe_json(recipe_data, recipe_slug)
        
        if not success:
            return False, None, "JSON_SAVE_FAILED"
        
        # Extract summary info
        name = recipe_data.get('name', 'N/A')
        ingredients = len(recipe_data.get('recipeIngredient', []))
        print(f"  SUCCESS: {name} ({ingredients} ingredients) -> {filename}")
        
        return True, filename, None
        
    except Exception as e:
        print(f"  ERROR: {type(e).__name__}: {str(e)[:100]}")
        return False, None, f"ERROR_{type(e).__name__}"

def scrape_all_recipes():
    """Main recipe scraping function"""
    print("="*80)
    print("WOOLWORTHS RECIPE SCRAPER")
    print("="*80)
    
    # Step 1: Fetch recipe sitemap
    sitemap_url = "https://www.woolworths.com.au/sitemap-recipes.xml"
    print(f"\nFetching recipe sitemap: {sitemap_url}")
    
    content = fetch_sitemap(sitemap_url)
    
    if not content:
        print("FAIL: Could not fetch recipe sitemap")
        return
    
    # Step 2: Parse recipe URLs
    print("Parsing recipe URLs...")
    current_recipes = parse_recipe_urls(content)
    print(f"SUCCESS: Found {len(current_recipes)} recipes in sitemap")
    
    # Step 3: Load or create tracker
    existing_df = load_or_create_tracker()
    
    if existing_df is not None:
        # Compare with existing tracker to find new recipes
        existing_slugs = set(existing_df['recipe_slug'].values)
        current_slugs = set([r['recipe_slug'] for r in current_recipes])
        
        new_slugs = current_slugs - existing_slugs
        
        if new_slugs:
            print(f"INFO: Found {len(new_slugs)} new recipes")
            # Add new recipes to tracker
            new_recipes = [r for r in current_recipes if r['recipe_slug'] in new_slugs]
            new_df = pd.DataFrame(new_recipes)
            df = pd.concat([existing_df, new_df], ignore_index=True)
        else:
            print("INFO: No new recipes found")
            df = existing_df
    else:
        # First run - create new tracker
        df = pd.DataFrame(current_recipes)
        print(f"INFO: Created new tracker with {len(df)} recipes")
    
    # Save initial tracker
    save_tracker(df)
    
    # Step 4: Scraping iterations
    for iteration_attempt in range(1, MAX_ITERATIONS + 1):
        print(f"\n{'='*80}")
        print(f"ITERATION {iteration_attempt}/{MAX_ITERATIONS}")
        print("="*80)
        
        # Filter unscraped recipes
        unscraped = df[df['downloaded'] == False]
        
        if unscraped.empty:
            print("SUCCESS: All recipes downloaded!")
            break
        
        print(f"Recipes to download: {len(unscraped)}")
        
        # Force IP rotation ONLY on 4th iteration
        force_rotate = (iteration_attempt == MAX_ITERATIONS)
        if force_rotate:
            print("NOTE: Final iteration - forcing IP rotation with retries")
        
        scraped_count = 0
        
        for idx, row in unscraped.iterrows():
            url = row['url']
            recipe_slug = row['recipe_slug']
            
            print(f"\n[{scraped_count + 1}/{len(unscraped)}] {recipe_slug}")
            
            # On final iteration, retry multiple times with forced rotation
            if iteration_attempt == MAX_ITERATIONS:
                recipe_success = False
                
                for retry in range(1, MAX_RETRIES_ON_FINAL_ITERATION + 1):
                    print(f"  Retry {retry}/{MAX_RETRIES_ON_FINAL_ITERATION}")
                    
                    success, filename, error = scrape_single_recipe(
                        url,
                        recipe_slug,
                        force_rotate=True
                    )
                    
                    if success:
                        df.loc[idx, 'downloaded'] = True
                        df.loc[idx, 'download_status'] = 'SUCCESS'
                        df.loc[idx, 'error_msg'] = ''
                        df.loc[idx, 'timestamp'] = datetime.now().isoformat()
                        scraped_count += 1
                        recipe_success = True
                        break
                    else:
                        print(f"  FAIL (retry {retry}): {error}")
                        
                        # Check for credit exhaustion
                        if 'insufficient credits' in (error or '').lower() or '403' in (error or ''):
                            print("\nERROR: CREDITS EXHAUSTED - Stopping scrape")
                            df.loc[idx, 'download_status'] = 'ERROR'
                            df.loc[idx, 'error_msg'] = 'CREDIT_EXHAUSTED'
                            save_tracker(df)
                            return
                        
                        if retry < MAX_RETRIES_ON_FINAL_ITERATION:
                            time.sleep(random.uniform(2, 4))
                
                if not recipe_success:
                    df.loc[idx, 'downloaded'] = False
                    df.loc[idx, 'download_status'] = 'ERROR'
                    df.loc[idx, 'error_msg'] = f'FAILED_AFTER_{MAX_RETRIES_ON_FINAL_ITERATION}_RETRIES'
                    print(f"  PERMANENT FAIL: Could not download after {MAX_RETRIES_ON_FINAL_ITERATION} retries")
            
            else:
                # Iterations 1-3: Single attempt
                success, filename, error = scrape_single_recipe(
                    url,
                    recipe_slug,
                    force_rotate=False
                )
                
                if success:
                    df.loc[idx, 'downloaded'] = True
                    df.loc[idx, 'download_status'] = 'SUCCESS'
                    df.loc[idx, 'error_msg'] = ''
                    df.loc[idx, 'timestamp'] = datetime.now().isoformat()
                    scraped_count += 1
                else:
                    df.loc[idx, 'downloaded'] = False
                    df.loc[idx, 'download_status'] = 'ERROR'
                    df.loc[idx, 'error_msg'] = error or 'Unknown'
                    print(f"  FAIL: {error}")
                    
                    # Check for credit exhaustion
                    if 'insufficient credits' in (error or '').lower() or '403' in (error or ''):
                        print("\nERROR: CREDITS EXHAUSTED - Stopping scrape")
                        save_tracker(df)
                        return
            
            # Save progress every 5 recipes
            if (scraped_count % 5 == 0) and scraped_count > 0:
                save_tracker(df)
                print(f"\n  Progress saved ({scraped_count} recipes)\n")
            
            # Rate limit
            time.sleep(random.uniform(1, 2))
        
        # Save after each iteration
        save_tracker(df)
        print(f"\nIteration {iteration_attempt} complete: {scraped_count} recipes downloaded")
        
        if iteration_attempt == MAX_ITERATIONS:
            print(f"NOTE: Reached maximum iterations ({MAX_ITERATIONS})")
            break
    
    # Final summary
    print("\n" + "="*80)
    print("SCRAPING COMPLETE")
    print("="*80)
    
    success_count = len(df[df['downloaded'] == True])
    failed_count = len(df[df['downloaded'] == False])
    
    print(f"Total recipes: {len(df)}")
    print(f"Successfully downloaded: {success_count}")
    print(f"Failed: {failed_count}")
    print(f"\nRecipe files saved in: {OUTPUT_DIR}/")
    print(f"Tracker file: {OUTPUT_DIR}/{CSV_FILENAME}")
    print("="*80)

if __name__ == "__main__":
    scrape_all_recipes()